In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%bash
unzip -o nto-ai-25-26-individual-baseline-main.zip -d .
unzip -o stage1_individual_data.zip -d .


Archive:  nto-ai-25-26-individual-baseline-main.zip
683d0f35849b2dd40572b712ab47e64aa6f17c54
   creating: ./nto-ai-25-26-individual-baseline-main/
  inflating: ./nto-ai-25-26-individual-baseline-main/.editorconfig  
  inflating: ./nto-ai-25-26-individual-baseline-main/.gitignore  
  inflating: ./nto-ai-25-26-individual-baseline-main/.pre-commit-config.yaml  
  inflating: ./nto-ai-25-26-individual-baseline-main/Makefile  
  inflating: ./nto-ai-25-26-individual-baseline-main/README.md  
   creating: ./nto-ai-25-26-individual-baseline-main/data/
   creating: ./nto-ai-25-26-individual-baseline-main/data/interim/
 extracting: ./nto-ai-25-26-individual-baseline-main/data/interim/.gitkeep  
   creating: ./nto-ai-25-26-individual-baseline-main/data/processed/
 extracting: ./nto-ai-25-26-individual-baseline-main/data/processed/.gitkeep  
   creating: ./nto-ai-25-26-individual-baseline-main/data/raw/
 extracting: ./nto-ai-25-26-individual-baseline-main/data/raw/.gitkeep  
   creating: ./nto-ai-2

In [ ]:
!find . -maxdepth 3 -type f -name "*.csv"


./users.csv
./sample_submission.csv
./test.csv
./book_descriptions.csv
./book_genres.csv
./train.csv
./genres.csv
./books.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_test.csv
./sample_data/mnist_test.csv
./sample_data/california_housing_train.csv


In [ ]:
%%bash
mkdir -p nto-ai-25-26-individual-baseline-main/data/raw
cp ./*.csv nto-ai-25-26-individual-baseline-main/data/raw/
ls nto-ai-25-26-individual-baseline-main/data/raw/


book_descriptions.csv
book_genres.csv
books.csv
genres.csv
sample_submission.csv
test.csv
train.csv
users.csv


In [ ]:
pip install -q pandas scikit-learn lightgbm joblib transformers sentencepiece torch tqdm pyarrow


In [ ]:
%%bash
cd nto-ai-25-26-individual-baseline-main
python -m src.baseline.prepare_data

Data Preparation Pipeline
Loading data...
Filtered training data: 268581 -> 156179 rows (only has_read=1)
Data loaded. Merging datasets...
Merged data shape: (159073, 15)
Starting feature engineering pipeline...
Adding genre features...
Adding text features (TF-IDF)...
Loading existing vectorizer from /content/nto-ai-25-26-individual-baseline-main/output/models/tfidf_vectorizer.pkl
Added 500 TF-IDF features.
Adding text features (BERT embeddings)...
Computing BERT embeddings (this may take a while)...
Using device: cuda
GPU memory limited to 75% of available memory
Saved BERT embeddings to /content/nto-ai-25-26-individual-baseline-main/output/models/bert_embeddings.pkl
Added 768 BERT features.
Handling missing values...
Feature engineering complete.

Saving processed data to /content/nto-ai-25-26-individual-baseline-main/data/processed/processed_features.parquet...
Processed data saved successfully!

Data preparation complete!
  - Train rows: 156,179
  - Test rows: 2,894
  - Total feat

2025-11-20 18:09:17.536285: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763662157.554470    3701 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763662157.560472    3701 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763662157.575989    3701 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763662157.576011    3701 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763662157.576016    3701 computation_placer.cc:177] computation placer alr

In [ ]:
%%bash
cd nto-ai-25-26-individual-baseline-main
python -m src.baseline.train

Loading prepared data from /content/nto-ai-25-26-individual-baseline-main/data/processed/processed_features.parquet...
Loaded 159,073 rows with 1284 features

Performing temporal split with ratio 0.8...
Split date: 2020-09-27 16:17:15
Train split: 124,944 rows
Validation split: 31,235 rows
Max train timestamp: 2020-09-27 16:17:15
Min validation timestamp: 2020-09-27 16:51:29
✅ Temporal split validation passed: all validation timestamps are after train timestamps

Computing aggregate features on train split only...
Adding aggregate features...
Adding aggregate features...
Handling missing values...
Handling missing values...
Handling missing values...
Training features: 1284

Training LightGBM model...

Validation RMSE: 2.8890, MAE: 2.1046
Model saved to /content/nto-ai-25-26-individual-baseline-main/output/models/lgb_model.txt

Training complete.


In [ ]:
%%bash
cd nto-ai-25-26-individual-baseline-main
python -m src.baseline.predict

Loading prepared data from /content/nto-ai-25-26-individual-baseline-main/data/processed/processed_features.parquet...
Loaded 159,073 rows with 1284 features
Train set: 156,179 rows
Test set: 2,894 rows

Computing aggregate features on all train data...
Adding aggregate features...
Handling missing values...
Handling missing values...
Prediction features: 1284

Loading model from /content/nto-ai-25-26-individual-baseline-main/output/models/lgb_model.txt...
Generating predictions...

Submission file created at: /content/nto-ai-25-26-individual-baseline-main/output/submissions/submission.csv
Predictions: min=3.7602, max=8.8680, mean=7.7151


In [ ]:
%%bash
cd nto-ai-25-26-individual-baseline-main
python -m src.baseline.validate

Validating submission file...
✅ Length check passed.
✅ No missing values check passed.
✅ (user_id, book_id) pair matching check passed.
✅ Prediction range [0, 10] check passed.

Validation successful! The submission file appears to be in the correct format.


In [ ]:
import pandas as pd
sub_path = "nto-ai-25-26-individual-baseline-main/output/submissions/submission.csv"
submission = pd.read_csv(sub_path)
submission.head()

,user_id,book_id,rating_predict
0,281,2461928,8.594814
1,1250,31957,7.555385
2,4241,196603,8.051517
3,5140,468894,8.329851
4,7781,2141951,7.492607


УЛУЧШЕНИЕ НОМЕР 1



In [ ]:
%%writefile nto-ai-25-26-individual-baseline-main/src/baseline/per_user_split.py
import pandas as pd

def per_user_last_interaction_split(df, user_col="user_id", time_col="timestamp"):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])

    val_idx = []
    train_idx = []

    for user, group in df.groupby(user_col):
        group = group.sort_values(time_col)
        val_idx.append(group.index[-1])
        train_idx.extend(group.index[:-1])

    return train_idx, val_idx


Overwriting nto-ai-25-26-individual-baseline-main/src/baseline/per_user_split.py


In [ ]:
import re

train_py_path = "nto-ai-25-26-individual-baseline-main/src/baseline/train.py"

with open(train_py_path, "r", encoding="utf-8") as f:
    code = f.read()

code = re.sub(
    r"split_date[\s\S]*?temporal_split_by_date.*?\)",
    'from .per_user_split import per_user_last_interaction_split\n'
    'print("работаем")\n'
    'train_idx, val_idx = per_user_last_interaction_split(train_set)',
    code
)

with open(train_py_path, "w", encoding="utf-8") as f:
    f.write(code)

print("train.py сделано")


train.py сделано


In [ ]:
%%bash
cd nto-ai-25-26-individual-baseline-main
python -m src.baseline.train

Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 159, in _get_module_details
  File "<frozen importlib._bootstrap_external>", line 1133, in get_code
  File "<frozen importlib._bootstrap_external>", line 1063, in source_to_code
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/content/nto-ai-25-26-individual-baseline-main/src/baseline/train.py", line 15
    from .temporal_split import get_from .per_user_split import per_user_last_interaction_split
                                         ^
SyntaxError: invalid syntax


CalledProcessError: Command 'b'cd nto-ai-25-26-individual-baseline-main\npython -m src.baseline.train\n'' returned non-zero exit status 1.

In [ ]:
train_py_path = "nto-ai-25-26-individual-baseline-main/src/baseline/train.py"
with open(train_py_path, "r", encoding="utf-8") as f:
  print(f.read())

"""
Main training script for the LightGBM model.

Uses temporal split with absolute date threshold to ensure methodologically
correct validation without data leakage from future timestamps.
"""

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

from . import config, constants
from .features import add_aggregate_features, handle_missing_values
from .temporal_split import get_from .per_user_split import per_user_last_interaction_split
print("работаем")
train_idx, val_idx = per_user_last_interaction_split(train_set)

    # Split data
    train_split = train_set[train_mask].copy()
    val_split = train_set[val_mask].copy()

    print(f"Train split: {len(train_split):,} rows")
    print(f"Validation split: {len(val_split):,} rows")

    # Verify temporal correctness
    max_train_timestamp = train_split[constants.COL_TIMESTAMP].max()
    min_val_timestamp = val_split[constants.COL_TIMESTAMP].min()
    print(f"M